In [0]:
%pip install -U transformers torch torchvision mlflow accelerate bitsandbytes vllm==0.7.0

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
import torch
import transformers
import pandas as pd
import json
import os
import yaml

from transformers import AutoTokenizer, AutoProcessor, MllamaForConditionalGeneration
from vllm import LLM, SamplingParams

In [0]:
from huggingface_hub import login

login()

In [0]:
model_path = 'meta-llama/Llama-3.2-11B-Vision-Instruct'
#model_path = "meta-llama/Llama-3.2-90B-Vision-Instruct"

model_cache_path = f"/local_disk0/models_cache/{model_path}"

In [0]:
from huggingface_hub import snapshot_download

snapshot_location = snapshot_download(repo_id=model_path, 
                                      local_dir=model_cache_path,
                                      ignore_patterns="*.pth")
snapshot_location

In [0]:
from mlflow.models.signature import infer_signature, ModelSignature
import requests
from PIL import Image
import io
import base64

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

example_image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
example_image = Image.open(requests.get(example_image_url, stream=True).raw)
example_image_base64 = pillow_image_to_base64_string(example_image)

input_example = pd.DataFrame().from_records([{"user_prompt": "describe the image and note the objects in the image", "image": example_image_base64}])
params = {"max_new_tokens": 256, "temperature": 0.01, "top_p":0.1}
output_example = pd.DataFrame().from_records([{"output_text": "this is an example output"}])

signature = infer_signature(input_example, output_example, params)
print(signature)

In [0]:
ds_model_path = os.path.join(os.getcwd(), "llama_vlm_model.py")
config_path = os.path.join(os.getcwd(), "inference_config.yml")

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        "model",
        python_model=ds_model_path,
        model_config=config_path,
        artifacts={"model_path": model_cache_path},
        input_example=input_example,
        signature=signature,
        pip_requirements=["transformers", "torch", "torchvision", "accelerate", "mlflow==2.20.0", "bitsandbytes", "vllm==0.7.0"]
    )

In [0]:
model_info.model_uri

## Load and test the model

In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import pandas as pd
from PIL import Image
import mlflow
import io
import os
import base64

def reduce_image_size(img, factor=4):
    width, height = img.size
    new_size = (width // factor, height // factor)
    return img.resize(new_size)

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# example_image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# example_image = Image.open(requests.get(example_image_url, stream=True).raw)

example_image_url = "/Volumes/uc_sriharsha_jana/test_db/shjdata/test_image.png"
example_image = Image.open(example_image_url).convert("RGB")

# example_image_resized = reduce_image_size(example_image)
example_image_base64 = pillow_image_to_base64_string(example_image)

input_example = pd.DataFrame().from_records([{"user_prompt": "describe the image and note the objects in the image", "image": example_image_base64}])

In [0]:
mlflow.set_registry_uri("databricks-uc")

# model_uri = 'runs:/ff6e3a64a71f44d9a11974bc05b9e179/model'
model_uri = "models:/uc_sriharsha_jana.test_db.llama-11b-vlm-model/3"
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
output = loaded_model.predict(input_example, params={"max_new_tokens": 512, "temperature": 0.01, "top_p":0.1})
print(output.iloc[0]["output_text"])

## Register the version to Model Registry

In [0]:
mlflow.register_model(model_uri, 
                      name="uc_sriharsha_jana.test_db.llama-11b-vlm-model",
                      tags={"type":"transformers-config"})